# Part 3 Training Notebook

This notebook is assigned to part `3`.
It reads the shared split manifests and trains the `48` configurations assigned to this part.


## Optional Google Colab Bootstrap

If you open this notebook directly from GitHub in Google Colab, run the next cell after:

1. Setting `GITHUB_REPO_URL`
2. Changing `RUN_COLAB_BOOTSTRAP` to `True`

Leave the cell as-is for local Jupyter use.


In [ ]:
import os
import subprocess
from pathlib import Path

RUN_COLAB_BOOTSTRAP = False
GITHUB_REPO_URL = ""
USE_GOOGLE_DRIVE = True
COLAB_REPO_DIR = "/content/drive/MyDrive/hi-192-dental-xray"

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and RUN_COLAB_BOOTSTRAP:
    if USE_GOOGLE_DRIVE:
        drive.mount("/content/drive")

    repo_root = Path(COLAB_REPO_DIR if USE_GOOGLE_DRIVE else "/content/hi-192-dental-xray")
    repo_root.parent.mkdir(parents=True, exist_ok=True)

    if not (repo_root / ".git").exists():
        if not GITHUB_REPO_URL.strip():
            raise ValueError("Set GITHUB_REPO_URL before running the Colab bootstrap cell.")
        subprocess.run(["git", "clone", GITHUB_REPO_URL, str(repo_root)], check=True)

    os.chdir(repo_root)
    print(f"Colab working directory set to: {repo_root}")
else:
    print(f"Current working directory: {Path.cwd()}")


In [ ]:
from pathlib import Path
import sys

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "dental_opg_experiment.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. In Colab, run the bootstrap cell first or clone the repo manually."
    )

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import dental_opg_experiment as doe


In [ ]:
from pathlib import Path

DATASET_ROOT = None
OUTPUTS_DIR = None

resolved_dataset_root = doe.resolve_dataset_root(DATASET_ROOT)
resolved_outputs_dir = doe.resolve_outputs_dir(OUTPUTS_DIR)

print(f"Dataset root: {resolved_dataset_root}")
print(f"Outputs dir: {resolved_outputs_dir}")


In [ ]:
PART_ID = 3
OVERWRITE_EXISTING = False
ONLY_CONFIG_IDS = []  # Example: ["base_cnn_drop_0p2_batch_16_dense_64"]
STOP_ON_ERROR = False


In [ ]:
configs = doe.load_part_configs(PART_ID)
print(f"Part {PART_ID} has {len(configs)} configs.")

for config in configs[:5]:
    print(config)


In [ ]:
import json

output_root = doe.resolve_outputs_dir(OUTPUTS_DIR) / f"part_{PART_ID}"
output_root.mkdir(parents=True, exist_ok=True)

selected_configs = [cfg for cfg in configs if not ONLY_CONFIG_IDS or cfg["config_id"] in ONLY_CONFIG_IDS]
results = []

for config in selected_configs:
    run_dir = output_root / config["config_id"]
    metrics_path = run_dir / "metrics.json"

    if metrics_path.exists() and not OVERWRITE_EXISTING:
        print(f"Skipping completed run: {config['config_id']}")
        results.append(json.loads(metrics_path.read_text(encoding="utf-8")))
        continue

    print(
        f"Running {config['config_id']} -> {config['architecture']}, "
        f"dropout={config['dropout']}, batch={config['batch_size']}, dense={config['dense_units']}"
    )

    try:
        metrics = doe.train_single_config(
            config=config,
            part_id=PART_ID,
            outputs_dir=OUTPUTS_DIR,
            dataset_root=DATASET_ROOT,
        )
        results.append(metrics)
    except Exception as exc:
        print(f"Failed on {config['config_id']}: {exc}")
        if STOP_ON_ERROR:
            raise

print(f"Collected {len(results)} result rows.")


In [ ]:
summary_df = doe.aggregate_part_results(outputs_dir=OUTPUTS_DIR, part_id=PART_ID)
summary_path = doe.resolve_outputs_dir(OUTPUTS_DIR) / f"part_{PART_ID}" / "part_summary.csv"

if not summary_df.empty:
    summary_df.to_csv(summary_path, index=False)
    display(summary_df.sort_values("accuracy", ascending=False).head(10))
    print(f"Saved summary to: {summary_path}")
else:
    print("No completed runs yet.")
